# Tahap 2b — Uji signifikansi AUC dan akurasi label dominan (CPU, < 1 menit)
Membaca prediksi test yang sudah disimpan notebook 20 di folder `stage2`. Tidak ada pelatihan model.
Pertanyaan: apakah keunggulan tipis Hybrid/BERT atas TF-IDF pada **AUC rata-rata** dan **akurasi label dominan** nyata, atau hanya kebetulan?

In [ ]:
import os, glob, re, numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
OUT4='/content/drive/MyDrive/JURNAL-OUTPUTS/laptop_jurnal'; PREP4=f'{OUT4}/prep'; S2=f'{OUT4}/stage2'
from sklearn.metrics import roc_auc_score
from scipy.stats import binomtest
y_te=np.load(f'{PREP4}/y_te_v3.npy'); Yb=(y_te>=0.5).astype(int); dom_true=np.argmax(y_te,1)
names={'TFIDF_Ridge':'TFIDF-Ridge','TFIDF_LinearSVR':'TFIDF-LinearSVR','TFIDF_LogReg':'TFIDF-LogReg','TFIDF_SVD_RBFSVR':'TFIDF-SVD-RBFSVR','BERT_only_IndoBERT_':'BERT-only','Hybrid_BERT_SVR':'Hybrid'}
P={}
for k,v in names.items():
    f=f'{S2}/test_pred_{k}.npy'
    if os.path.exists(f): P[v]=np.load(f)
    else: print('TIDAK ADA:',f, '| file yang ada:',[os.path.basename(x) for x in glob.glob(S2+'/test_pred_*.npy')])
print(list(P))

In [ ]:
def mauc(idx,Pm):
    a=[]
    for k in range(4):
        y=Yb[idx,k]
        if y.min()==y.max(): return np.nan
        a.append(roc_auc_score(y,Pm[idx,k]))
    return np.mean(a)
rng=np.random.default_rng(42); n=len(y_te); B=5000
boots=[rng.integers(0,n,n) for _ in range(B)]
rows=[]
H=P['Hybrid']; dH=(np.argmax(H,1)==dom_true)
for name,Pm in P.items():
    if name=='Hybrid': continue
    d_auc=np.array([mauc(i,H)-mauc(i,Pm) for i in boots]); d_auc=d_auc[~np.isnan(d_auc)]
    dm=(np.argmax(Pm,1)==dom_true)
    d_dom=np.array([dH[i].mean()-dm[i].mean() for i in boots])
    b=int((dH&~dm).sum()); c=int((~dH&dm).sum())
    p=binomtest(b,b+c,0.5).pvalue if b+c>0 else 1.0
    rows.append({'Model (pembanding)':name,
      'ΔAUC (Hybrid−model)':mauc(np.arange(n),H)-mauc(np.arange(n),Pm),'AUC_CI_low':np.percentile(d_auc,2.5),'AUC_CI_high':np.percentile(d_auc,97.5),
      'ΔAkurasi dominan':dH.mean()-dm.mean(),'Dom_CI_low':np.percentile(d_dom,2.5),'Dom_CI_high':np.percentile(d_dom,97.5),
      'McNemar_dominan_p':p,'Hybrid_benar_model_salah':b,'Hybrid_salah_model_benar':c})
res=pd.DataFrame(rows).set_index('Model (pembanding)')
pd.set_option('display.width',250); pd.set_option('display.max_columns',20)
print(res.round(4).to_string())
res.to_csv(f'{S2}/stage2b_auc_dominant_vs_hybrid.csv')
print('\nCI yang tidak melewati 0 => selisih signifikan (95%).')

Kirim screenshot tabel di atas (atau file `stage2b_auc_dominant_vs_hybrid.csv` di folder `stage2`).